# Task 4 – Szenarien für die Prüfung

Was passiert im Netz am Ende von Teil 4 (Windpark an Bus 1, PV-Freiflächenanlage an Bus 12, Wohngebiet mit PV-Haushalten und Heimspeichern, rONT-Regelung, S1–S3 offen), wenn sich Wetter, Verbrauch oder Netz ändern? Jedes Szenario besteht aus

- **Frage** und **Annahme** (was genau verändert wird),
- **Rechnung** gegen einen Referenzlauf mit unverändertem Netz in derselben typischen Woche,
- **Spannungsprofil**: Spannung von Bus zu Bus vom 110-kV-Netz bis zum Strangende und der Spannungsfall je Abschnitt (HS/MS-Trafo, MS-Kabel, Ortsnetztrafo mit rONT-Stufe, NS-Kabel),
- **Kennzahlentabelle** und **Zeitverlauf**,
- **Befund** mit physikalischer Erklärung.

Für Änderungen am Wetter oder am Verbrauch gibt es ein neues Werkzeug: **Profile kopieren und bearbeiten** (`profilaenderung.py`, Abschnitt 1). Damit lässt sich die Leistung einzelner Elemente einmalig oder periodisch auf einen festen Wert setzen, mit einem Faktor skalieren oder um einen Betrag erhöhen.

Voraussetzung: Die Notebooks Teil 1 bis 4a wurden ausgeführt (Simulation `w4_kombi_W1_P12`). Gerechnet wird jeweils nur die typische Woche, um die es geht (168 Lastflüsse je Variante); alle Varianten heißen `w4_p_…` und überschreiben keine Simulation aus Teil 1 bis 8.

| Nr. | Szenario | Woche |
|---|---|---|
| 2 | Referenz und Spannungsprofil lesen | Januar, Juli |
| 3 | Mittags ziehen Wolken auf | Juli |
| 4 | Die Haushalte verbrauchen mehr als üblich | Januar |
| 5 | Dunkelflaute: 100 / 60 / 30 / 0 % der normalen Einspeisung | Januar |
| 6 | Starkwind nachts bei Schwachlast, größerer Windpark | Juni |
| 7 | Sturmabschaltung des Windparks bei Volllast | Januar |
| 8 | Sonniger Sonntag mit niedriger Last, heute und Zukunft | Juli |
| 9 | S1 wird bei viel Wind geschlossen | Dezember |
| 10 | Ausfall von Line 2-3 mit Wiederversorgung | Januar |
| 11 | Ausfall von Trafo 0-1 | Januar |
| 12 | Der Stufenschalter fällt aus | Januar, Juli |
| 13 | Windpark mit cos φ = 0,95 untererregt | Januar |
| 14 | Spannung im 110-kV-Netz 1,00 bzw. 1,05 pu | Januar, Juli |
| 15 | Zusätzliche Einspeisung an Bus 8 (Line 3-8) | Juli |
| 16 | Heimspeicher: ohne, normal, doppelt so groß | Juli |
| 17 | Übersicht |  |

## Importe

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))   # Module liegen in task4/src, das Notebook in task4/notebooks

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import simulation as sim
import profilaenderung as pa
import pruefung as pr
from pruefung import WIND, PV_ALLE, PV_HAUSHALTE, HAUSHALTE
from elements import Generator, Profile

import warnings
warnings.simplefilter(action="ignore", category=FutureWarning)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)

sim.ONLY_RUN_IF_CHANGED = True

## 1 Werkzeug: Profile kopieren und bearbeiten

Jedes Element (Last oder Erzeuger) bekommt seine Leistung aus einem Profil, z. B. alle PV-Anlagen aus dem PV-Profil. Eine **Profiländerung** bearbeitet eine Kopie dieses Profils nur für die ausgewählten Elemente; alle anderen Elemente behalten das Original.

```python
pa.immer(elemente, faktor=… | wert_mw=… | plus_mw=…, monate=None)
pa.einmal(elemente, …, datum="2025-07-23", von=11, bis=14)          # oder monat="JULY", wochentag=2
pa.periodisch(elemente, …, alle_h=24, start_h=17, dauer_h=5)        # jeden Tag 17–22 Uhr
```

| Parameter | Bedeutung |
|---|---|
| `elemente` | regulärer Ausdruck auf den Elementnamen, z. B. `"^Windpark$"`, `"^LV PV"` (alle PV-Haushalte), `"^LV Load R"` (Haushalte im Wohngebiet) |
| `faktor` | Leistung × Faktor (0,3 = 30 % der normalen Leistung) |
| `wert_mw` | Leistung fest auf diesen Wert in MW (je Element) |
| `plus_mw` | Leistung + Betrag in MW (je Element) |
| `einmal` | ein Tag, Stunden `von` bis `bis` (ohne Tag: ganze Woche des Monats) |
| `periodisch` | alle `alle_h` Stunden für `dauer_h` Stunden, ab Stunde `start_h` der Woche (Montag 0 Uhr = 0): täglich = 24, wöchentlich = 168 |

Die Änderungen werden mit `s.changeSetting("profile_edits", [...])` an eine Simulation gehängt (in `pr.szenario` über `aenderungen=`). Sie gehören zum Fingerabdruck: Gespeicherte Ergebnisse werden nur wiederverwendet, wenn die Änderungen gleich sind. Die Blindleistung folgt über den cos φ, die Heimspeicher rechnen ihren Fahrplan mit der geänderten Leistung.

Kurznamen für die Elemente in diesem Notebook:

In [ ]:
pd.DataFrame({"Muster": [WIND, PV_ALLE, PV_HAUSHALTE, HAUSHALTE],
              "Elemente": ["Windpark (30 MW, Bus 1)", "PV-Freifläche (Bus 12) und alle PV-Anlagen der Haushalte",
                           "nur PV-Anlagen der Haushalte", "Haushaltslasten im Wohngebiet (R1, R11, R15 … R18)"]},
             index=["WIND", "PV_ALLE", "PV_HAUSHALTE", "HAUSHALTE"])

Daten der typischen Wochen (für `einmal(datum=…)`):

In [ ]:
pa.typische_wochen()

**Beispiele** (ohne Lastfluss, nur die bearbeitete Kopie des Profils):

In [ ]:
b = pr.basis()
beispiel = [pa.immer(WIND, faktor=0.3, name="Dunkelflaute 30 %")]
pr.profil_vorschau(b, WIND, beispiel, monat="JANUARY", titel="Windpark: immer 30 % der normalen Leistung")

beispiel = [pa.einmal(PV_ALLE, faktor=0.2, datum="2025-07-23", von=11, bis=15, name="Wolken")]
pr.profil_vorschau(b, PV_ALLE, beispiel, monat="JULY", titel="PV: einmal am 23.07. von 11 bis 15 Uhr nur 20 %")

beispiel = [pa.periodisch(HAUSHALTE, plus_mw=0.02, alle_h=24, start_h=17, dauer_h=4, name="Abendspitze")]
pr.profil_vorschau(b, HAUSHALTE, beispiel, monat="JANUARY", tabelle="load",
                   titel="Haushalte: täglich 17–21 Uhr je Last + 20 kW (periodisch)")

Kontrolle, welche Elemente ein Muster trifft:

In [ ]:
pa.betroffene_elemente(b, [pa.immer(WIND, faktor=1), pa.immer(PV_ALLE, faktor=1)])

## 2 Ausgangsnetz und Referenzläufe

Ausgangsnetz ist das Netz am Ende von Teil 4 (`pr.basis()`: Kombination aus Teil 4a mit PV-Haushalten, Variante „heute“). Für jede typische Woche, die in einem Szenario vorkommt, wird es einmal unverändert gerechnet; alle Varianten werden mit diesem Referenzlauf verglichen.

In [ ]:
b = pr.basis()
REF = {m: pr.szenario(f"ref_{m.lower()}", b, monate=[m]) for m in ("JANUARY", "JUNE", "JULY", "DECEMBER")}
pr.kennzahlen({f"Referenz {m.title()}": s for m, s in REF.items()})

**Referenz:** In allen vier Wochen bleiben alle Busse im Planungsband 0,95 … 1,05 pu, kein Betriebsmittel ist über 100 %. Im Winter (Januar, Dezember) bestimmt die Abendspitze das Netz: U min 0,952 pu an Bus 11, Trafo 0-1 bis 88 % (bei Starkwind in Rückspeisung). Im Sommer (Juni, Juli) bestimmt die Mittagssonne das Wohngebiet: Rückspeisung über Ortsnetztrafo R in 50 Stunden, U max 1,049 pu an R15, der rONT stuft dann nach unten.

### Spannung von Punkt zu Punkt

Die Kennzahlen oben zeigen nur den kleinsten und größten Wert im ganzen Netz. Wo die Spannung verloren geht, zeigt das **Spannungsprofil**: die Spannung an jedem Bus auf dem Weg vom 110-kV-Netz bis zum Strangende, zu einer bestimmten Stunde. Der Hintergrund zeigt, welches Betriebsmittel zwischen zwei Bussen liegt; oben steht die Länge des Abschnitts. Die Balken darunter fassen den Spannungsfall je Abschnittstyp zusammen:

- **HS/MS-Trafo:** Spannungsfall im 110/20-kV-Trafo (wächst mit dem Strom).
- **MS-Leitung:** Spannungsfall auf den 20-kV-Kabeln bis zum Anschlusspunkt.
- **Ortsnetztrafo:** Spannungsfall im 20/0,4-kV-Trafo **einschließlich der rONT-Stufe**. Ein negativer Wert heißt: Der rONT hebt die Spannung an (Stufe −1/−2); ein großer positiver Wert: Er senkt ab (Stufe +1/+2).
- **NS-Leitung:** Spannungsfall auf den 0,4-kV-Kabeln von der Sammelschiene bis zum Strangende.

Welche Stunde gezeigt wird, legt `wahl` fest; alle Varianten werden zur **selben Stunde** verglichen:

| `wahl` | Stunde |
|---|---|
| `"fall"` (Standard) | größter Spannungsfall auf den Kabeln des Strangs (hohe Last), bestimmt von der Variante mit dem größten Wert |
| `"steig"` | größte Spannungsanhebung auf den Kabeln (Rückspeisung) |
| `"min"` / `"max"` | niedrigste / höchste Spannung am Strangende |
| `"fall_je"`, `"min_je"`, … | wie oben, aber je Variante ihre eigene Stunde |
| `"2025-07-25 13:00"` | feste Stunde |

Im Ortsnetz ist `"fall"` meist aussagekräftiger als `"min"`: Der rONT hält das Strangende bei Last immer knapp über 0,95 pu, deshalb gibt es viele Stunden mit fast derselben niedrigsten Spannung; der größte Spannungsfall zeigt dagegen die Stunde mit der höchsten Belastung. Stränge: `"Abgang 1"` (bis Bus 11), `"Abgang 2"` (bis Bus 14), `"Wohngebiet"` (bis R15), `"Wohngebiet (R18)"`, `"Gewerbe"` (bis C12), `"Industrie"` (bis I2).

**Referenz Winter (Abendspitze) und Sommer (Mittag, Rückspeisung) im Wohngebiet:**

In [ ]:
ws = {"Januar, Abendspitze": REF["JANUARY"], "Juli, Mittag": REF["JULY"]}
pr.profil(ws, "Wohngebiet", wahl={"Januar, Abendspitze": "fall", "Juli, Mittag": "steig"},
          titel="Referenz Wohngebiet: Winterabend (größter Spannungsfall) und Sommermittag (größte Anhebung)")
pr.abschnitte(ws, "Wohngebiet", wahl={"Januar, Abendspitze": "fall", "Juli, Mittag": "steig"})

**Referenz MS-Netz und die anderen Teilnetze** (Januar, jeweils Stunde mit dem größten Spannungsfall):

In [ ]:
for strang in ("Abgang 1", "Abgang 2", "Gewerbe", "Industrie"):
    display(pr.abschnitte({strang: REF["JANUARY"]}, strang))
pr.profil({"Januar": REF["JANUARY"]}, "Abgang 1", titel="Referenz Abgang 1 (Januar, größter Spannungsfall)")

**Über die ganze Woche:** Spannung jedes Busses im Wohngebiet über die Zeit (Farbe). Unten liegen die Strangenden; dunkelblau heißt tiefe, rot hohe Spannung.

In [ ]:
pr.spannung_zeit_ort(REF["JANUARY"], "Wohngebiet", monat="JANUARY", titel="Referenz Januar: Spannung im Wohngebiet über die Woche")
pr.spannung_zeit_ort(REF["JULY"], "Wohngebiet", monat="JULY", titel="Referenz Juli: Spannung im Wohngebiet über die Woche")

**Befund Spannungsprofil (Referenz):**

- **Winterabend im Wohngebiet** (12.01., 18 Uhr): Vom 110-kV-Netz (1,030 pu) verliert der HS/MS-Trafo 2,3 Prozentpunkte, das Verbindungskabel zum Ortsnetztrafo nichts. Der rONT steht auf Stufe −1 und **hebt um 0,7 Prozentpunkte an**. Auf den nur rund 300 m NS-Kabeln bis R15 fallen dann **5,8 Prozentpunkte**; R15 liegt bei 0,956 pu. Der größte Teil des Spannungsfalls entsteht also auf den letzten Metern im Ortsnetz: Bei 0,4 kV fließt für dieselbe Leistung der 50-fache Strom wie bei 20 kV.
- **Sommermittag** (25.07., 13 Uhr): Die Richtung dreht sich. Auf den NS-Kabeln **steigt** die Spannung um 4,9 Prozentpunkte zum Strangende hin, weil die PV-Haushalte zurückspeisen. Der rONT steht auf Stufe +1 und senkt um 2,4 Prozentpunkte ab; R15 liegt bei 1,041 pu. Der Tiefpunkt liegt jetzt an der Sammelschiene R1 (0,993 pu), der Hochpunkt am Strangende.
- **Abgang 1** (Höchstlast): 2,4 Prozentpunkte im HS/MS-Trafo, 5,3 Prozentpunkte auf knapp 10 km MS-Kabel bis Bus 11 (0,953 pu). Fast der ganze Kabelfall entsteht auf den ersten beiden Abschnitten (Line 1-2 und 2-3, zusammen 7,2 km), danach ist die Kurve flach, weil dahinter kaum noch Last hängt.
- **Abgang 2** verliert auf seinen Kabeln nur 0,6 Prozentpunkte, **Gewerbe** 5,4 und **Industrie** 3,5 Prozentpunkte auf den NS-Kabeln.
- **Über die Woche** (Farbbilder): Im Winter liegt die tiefste Spannung jeden Abend am Strangende (unten blau), im Sommer ist das Strangende mittags am höchsten (rot) und die Sammelschiene am tiefsten.

## 3 Mittags ziehen Wolken auf

**Frage:** Was passiert, wenn an einem sonnigen Sommertag mittags Wolken aufziehen?

**Annahme:** Am sonnigsten Tag der Juliwoche liefern alle PV-Anlagen (Freifläche und Haushalte) von 11 bis 15 Uhr nur 20 % (**a**). Zusätzlich ein Wolkenzug über die ganze Woche: jede zweite Stunde nur 30 % (**b**). Die Stundenauflösung bildet nur den Mittelwert jeder Stunde ab; schnelle Wolkenzüge im Minutenbereich sind feiner, als das Modell auflöst.

In [ ]:
pv_tag = pr.leistung(REF["JULY"], PV_ALLE).resample("D").sum()
sonnig = pv_tag.idxmax().strftime("%Y-%m-%d")
print("sonnigster Tag der Juliwoche:", sonnig)
wolken = {"Referenz": REF["JULY"],
          "a) Wolken 11–15 Uhr (20 %)": pr.szenario("wolken_mittag", b, monate=["JULY"],
              aenderungen=[pa.einmal(PV_ALLE, faktor=0.2, datum=sonnig, von=11, bis=15, name="Wolken")]),
          "b) Wolkenzug (jede 2. Stunde 30 %)": pr.szenario("wolkenzug", b, monate=["JULY"],
              aenderungen=[pa.periodisch(PV_ALLE, faktor=0.3, alle_h=2, start_h=1, dauer_h=1, name="Wolkenzug")])}
tab = pr.kennzahlen(wolken)
tab

In [ ]:
pr.verlauf({k: wolken[k] for k in list(wolken)[:2]}, f"Wolken am {sonnig} von 11 bis 15 Uhr", tage=[sonnig])
pr.verlauf({k: wolken[k] for k in (list(wolken)[0], list(wolken)[2])}, "Wolkenzug: jede zweite Stunde 30 % PV", monat="JULY")

In [ ]:
# Wohngebiet in den Wolkenstunden: Austausch über Ortsnetztrafo R, Speicher, rONT
h = pd.date_range(f"{sonnig} 09:00", periods=9, freq="h")
zeilen = {}
for k in list(wolken)[:2]:
    r = wolken[k].results
    zeilen[k] = pd.DataFrame({"PV Haushalte [kW]": 1000 * pr.leistung(wolken[k], PV_HAUSHALTE).loc[h],
                              "Speicher [kW] (> 0 Laden)": 1000 * r.storage_p_mw.sum(axis=1).loc[h],
                              "Ortsnetztrafo R [kW] (< 0 Rückspeisung)": 1000 * r.trafo_p_mw[pr.ONT["R"]].loc[h],
                              "Stufe rONT R": r.tap_pos[pr.ONT["R"]].loc[h],
                              "U max NS [pu]": r.vm_pu[[c for c in r.vm_pu if c.startswith("LV Bus R")]].max(axis=1).loc[h]})
pd.concat(zeilen, axis=1).set_index(h.strftime("%H:%M")).round(3)

**Spannungsprofil im Wohngebiet** um 13 Uhr (Wolken) und um 15 Uhr (die Sonne kommt zurück, der rONT steht noch auf der Wolkenstufe):

In [ ]:
wk = {k: wolken[k] for k in list(wolken)[:2]}
pr.profil(wk, "Wohngebiet", wahl=f"{sonnig} 13:00", titel=f"Wohngebiet am {sonnig} um 13 Uhr")
display(pr.abschnitte(wk, "Wohngebiet", wahl=f"{sonnig} 13:00"))
pr.profil(wk, "Wohngebiet", wahl=f"{sonnig} 15:00", titel=f"Wohngebiet am {sonnig} um 15 Uhr")
pr.abschnitte(wk, "Wohngebiet", wahl=f"{sonnig} 15:00")

**Befund Wolken:**

- **Spannungsprofil:** Um 13 Uhr kehrt sich das Profil im Wohngebiet um. Ohne Wolken steigt die Spannung zum Strangende hin (NS-Kabel −4,9 Prozentpunkte, R15 1,041 pu), mit Wolken fällt sie leicht (+1,1 Prozentpunkte, R15 0,998 pu). Der rONT senkt mit Wolken nur noch 0,6 statt 2,4 Prozentpunkte ab. Um 15 Uhr, wenn die Sonne zurückkommt, steht er noch auf Stufe 0 (Trafo nur 0,1 Prozentpunkte), deshalb liegt R15 bei 1,045 statt 1,021 pu: Der Regler läuft der Sonne hinterher.

- **a) Wolken 11–15 Uhr:** Im Wohngebiet kippt der Fluss innerhalb einer Stunde von **360 kW Rückspeisung auf 63 bis 80 kW Bezug** (Sprung um gut 400 kW über Ortsnetztrafo R). Die Spannung an den Strangenden fällt von 1,04 auf 1,015 pu, der rONT stuft von +1 auf 0 zurück. Um 15 Uhr kommt die Sonne wieder: Die Spannung springt auf 1,045 pu, bevor der rONT wieder absenkt. Im MS-Netz bezieht Trafo 0-12 die fehlende PV-Leistung zusätzlich; die Spannungen ändern sich kaum, weil die PV-Freiflächenanlage an der Sammelschiene einspeist.
- **Speicher:** Sie sind um 10 Uhr schon voll und laden nicht. Sie entladen in den Wolkenstunden auch nicht, weil selbst 20 % PV noch über dem Verbrauch der Speicherhaushalte liegt. Der Wolkeneinbruch wird also nicht abgefedert; der Speicher reagiert nur auf den eigenen Haushalt.
- **b) Wolkenzug:** Rückspeisung aus dem Wohngebiet nur noch in 21 statt 50 Stunden, Trafo 0-12 und der Bezug aus dem 110-kV-Netz steigen leicht. Die Zahl der rONT-Stufenschritte bleibt gleich, weil das Modell nur Stundenwerte kennt: Wechselt die Bewölkung im Minutentakt, würde ein realer Regler ohne Verzögerung ständig schalten. Deshalb haben rONT eine Totzeit von 30 bis 60 s und zählen die Schaltspiele (Verschleiß).
- **Prüfungsantwort:** Wolken erzeugen schnelle Leistungssprünge in die Gegenrichtung. Spannungsprobleme entstehen kurzzeitig im Ortsnetz (Strangende), nicht im MS-Netz. Grenzen werden in diesem Netz nicht verletzt.

## 4 Die Haushalte verbrauchen mehr als üblich

**Frage:** Was passiert, wenn die Haushalte mehr Strom verbrauchen, z. B. in einer Kältewelle, durch Wärmepumpen oder durch E-Autos, die abends laden?

**Annahme (Januarwoche, Haushalte im Wohngebiet):**
- **a)** immer + 30 % (z. B. Kältewelle, mehr Heizlüfter und Wärmepumpen),
- **b)** täglich 17–21 Uhr + 60 % (z. B. E-Autos laden nach Feierabend),
- **c)** immer + 60 % (Belastungsgrenze des Wohngebiets).

In [ ]:
haushalte = {"Referenz": REF["JANUARY"],
             "a) immer +30 %": pr.szenario("last_plus30", b, monate=["JANUARY"],
                 aenderungen=[pa.immer(HAUSHALTE, faktor=1.3, name="+30 %")]),
             "b) täglich 17–21 Uhr +60 %": pr.szenario("last_abend60", b, monate=["JANUARY"],
                 aenderungen=[pa.periodisch(HAUSHALTE, faktor=1.6, alle_h=24, start_h=17, dauer_h=4, name="Abend +60 %")]),
             "c) immer +60 %": pr.szenario("last_plus60", b, monate=["JANUARY"],
                 aenderungen=[pa.immer(HAUSHALTE, faktor=1.6, name="+60 %")])}
tab = pr.kennzahlen(haushalte)
tab

In [ ]:
pr.verlauf(haushalte, "Mehr Verbrauch der Haushalte (Januarwoche)", monat="JANUARY")

**Spannungsprofil im Wohngebiet** zur Stunde mit dem größten Spannungsfall:

In [ ]:
pr.profil(haushalte, "Wohngebiet", wahl="fall", titel="Wohngebiet bei mehr Verbrauch (Stunde mit dem größten Spannungsfall)")
display(pr.abschnitte(haushalte, "Wohngebiet", wahl="fall"))
pr.spannung_zeit_ort(haushalte["c) immer +60 %"], "Wohngebiet", monat="JANUARY", titel="Haushalte +60 %: Spannung im Wohngebiet über die Woche")

**Befund mehr Verbrauch:**

- **Spannungsprofil** (11.01., 18 Uhr): Der Spannungsfall auf den NS-Kabeln wächst mit der Last von 5,6 (Referenz) auf 7,3 (+30 %) und 9,2 bis 9,6 Prozentpunkte (+60 %). Der rONT hebt dagegen an: −0,7 → −2,9 Prozentpunkte. Bei +60 % hebt er trotz Stufe −2 nur noch 2,2 Prozentpunkte an, weil der Spannungsfall im Trafo selbst mit dem Strom wächst. Ergebnis: R15 0,958 → 0,963 (+30 %, voll ausgeglichen) → 0,937/0,932 pu (+60 %, nicht mehr ausgleichbar).

- **a) +30 %:** Der rONT gleicht die Spannung aus (U min NS 0,952 pu), schaltet dafür häufiger (38 statt 24 Stufenschritte). Weil er alle Busse des Wohngebiets gleich anhebt, steigt die Sammelschiene R1 auf 1,044 pu. Ortsnetztrafo R ist mit 82 % ausgelastet, Kabel R1-R2 mit 86 %.
- **b) Abendspitze +60 % und c) immer +60 %:** Jetzt werden zuerst die **Betriebsmittel** zum Problem: Ortsnetztrafo R bis 103 bzw. 105 %, Kabel R1-R2 bis 108 bzw. 111 %, in 9 bis 10 Stunden überlastet. Zugleich fällt die Spannung an R15 auf 0,93 pu: Der rONT kann nicht weiter anheben, weil die Sammelschiene R1 sonst über 1,05 pu stiege. Der Spannungsunterschied innerhalb des Strangs ist größer als das Band; das kann ein Stufenschalter grundsätzlich nicht ausgleichen.
- **MS-Netz:** keine Änderung (U min 0,952 pu, Trafo 0-1 87,5 %). Das Wohngebiet ist für das MS-Netz klein.
- **Prüfungsantwort:** Mehr Verbrauch belastet zuerst Ortsnetztrafo und erstes Kabel thermisch; die Spannung hält der rONT, bis der Spannungsfall im Strang größer wird als die Bandbreite. Abhilfe: Lastmanagement (§ 14a EnWG, Laden zeitlich verteilen), Speicher abends entladen, größerer Trafo, Kabel verstärken.

## 5 Dunkelflaute

**Frage:** Was passiert, wenn Wind und Sonne über eine ganze Woche weniger liefern als normal?

**Annahme:** In der Januarwoche (höchste Last) liefern der Windpark und alle PV-Anlagen nur 60 %, 30 % bzw. 0 % ihrer normalen Leistung (Faktor auf beide Profile). 100 % ist der Referenzlauf.

In [ ]:
FAKTOREN = [1.0, 0.6, 0.3, 0.0]
flaute = {}
for f in FAKTOREN:
    if f == 1.0:
        flaute["100 % (Referenz)"] = REF["JANUARY"]
    else:
        flaute[f"{f:.0%}"] = pr.szenario(f"flaute_{int(f * 100)}", b, monate=["JANUARY"],
                                         aenderungen=[pa.immer(WIND, faktor=f), pa.immer(PV_ALLE, faktor=f)])
tab = pr.kennzahlen(flaute)
tab

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
x = [100 * f for f in FAKTOREN]
for ax, (zeile, titel) in zip(axes, [("U min MS [pu]", "niedrigste MS-Spannung [pu]"), ("Trafo 0-1 max [%]", "Trafo 0-1 max [%]"),
                                     ("max. Bezug 110 kV [MW]", "max. Bezug 110 kV [MW]"), ("Verluste [MWh]", "Verluste [MWh]")]):
    ax.plot(x, [pr.zahl(tab, zeile, k) for k in tab.columns], "o-")
    ax.set_title(titel, fontsize=10)
    ax.set_xlabel("Einspeisung [% der normalen]")
    ax.invert_xaxis()
    ax.grid(alpha=0.3)
axes[0].axhline(0.95, color="orange", ls="--", lw=1)
plt.tight_layout()
plt.show()
pr.verlauf({k: flaute[k] for k in ("100 % (Referenz)", "30%", "0%")}, "Dunkelflaute (Januarwoche)", monat="JANUARY")

**Spannungsprofil** in Abgang 1 und im Wohngebiet zur Stunde mit dem größten Spannungsfall. Die höchste Spannung liegt immer an der Sammelschiene; entscheidend ist, wie viel bis zum Strangende verloren geht:

In [ ]:
pr.profil(flaute, "Abgang 1", wahl="fall", titel="Dunkelflaute: Abgang 1 (Stunde mit dem größten Spannungsfall)")
display(pr.abschnitte(flaute, "Abgang 1", wahl="fall"))
pr.profil(flaute, "Wohngebiet", wahl="fall", titel="Dunkelflaute: Wohngebiet (Stunde mit dem größten Spannungsfall)")
pr.abschnitte(flaute, "Wohngebiet", wahl="fall")

**Befund Dunkelflaute:**

- **Spannungsprofil Abgang 1** (10.01., 18 Uhr): Der Spannungsfall auf den MS-Kabeln bleibt bei 5,3 Prozentpunkten, egal wie viel eingespeist wird. Nur der Fall im HS/MS-Trafo wächst von 2,4 auf 2,8 Prozentpunkte, weil der Trafo jetzt die ganze Last aus dem 110-kV-Netz holen muss. Grund: Windpark und PV hängen an den Sammelschienen, ihre Leistung fließt nicht durch die Kabel der Abgänge. Bus 11 sinkt nur von 0,953 auf 0,949 pu.
- **Spannungsprofil Wohngebiet** (12.01., 18 Uhr): Der Spannungsfall auf den NS-Kabeln steigt von 5,8 auf 6,2 Prozentpunkte. Ursache sind die **Heimspeicher**: Mit weniger PV am Tag sind sie abends leer und können den Abendbedarf nicht mehr decken. Ab 30 % Einspeisung ändert sich nichts mehr, die Speicher sind dann schon leer.

- **Spannung:** Mit weniger Einspeisung sinkt die niedrigste MS-Spannung nur wenig (0,952 → 0,949 pu bei 0 %). Die Erzeuger hängen an den Sammelschienen und stützen die Spannung in den Abgängen kaum.
- **Trafo 0-1 verhält sich nicht monoton:** 87,5 % bei voller Einspeisung (Rückspeisung bei Starkwind), 75,7 % bei 60 % (Wind und Last gleichen sich besser aus), dann wieder steigend auf 82,7 % bei 0 % (Höchstlast ohne Wind). Die Auslastung hängt am Betrag des Stroms; beide Richtungen belasten.
- **Austausch:** Rückspeisung ins 110-kV-Netz verschwindet ab 30 %, der höchste Bezug steigt auf das Niveau ohne Erzeuger. Die Verluste sind bei 60 % am kleinsten und bei 0 % am größten.
- **Prüfungsantwort:** Eine Dunkelflaute bringt das Netz in den Zustand ohne Erzeuger zurück, den es ohnehin beherrschen muss (Auslegung auf Höchstlast). Kritisch wird sie für die Energieversorgung (Bezug aus dem 110-kV-Netz), nicht für das Verteilnetz. Erzeuger ersetzen keine Netzkapazität.

## 6 Starkwind nachts bei Schwachlast

**Frage:** Was passiert, wenn nachts bei geringer Last der Wind voll weht? Und wenn der Windpark größer wäre?

**Annahme (Juniwoche, geringe Last):** Der Windpark speist jede Nacht von 0 bis 6 Uhr mit voller Leistung ein: **a)** 30 MW (installierte Leistung), **b)** 45 MW (Windpark um 50 % größer, nur nachts betrachtet).

In [ ]:
starkwind = {"Referenz": REF["JUNE"],
             "a) nachts 30 MW": pr.szenario("starkwind_30", b, monate=["JUNE"],
                 aenderungen=[pa.periodisch(WIND, wert_mw=30, alle_h=24, start_h=0, dauer_h=6, name="nachts volle Leistung")]),
             "b) nachts 45 MW": pr.szenario("starkwind_45", b, monate=["JUNE"],
                 aenderungen=[pa.periodisch(WIND, wert_mw=45, alle_h=24, start_h=0, dauer_h=6, name="größerer Windpark")])}
tab = pr.kennzahlen(starkwind)
tab

In [ ]:
pr.verlauf(starkwind, "Starkwind nachts (Juniwoche)", monat="JUNE")

**Spannungsprofil** in Abgang 1 in der Stunde mit der höchsten Rückspeisung (Windpark an Bus 1):

In [ ]:
t_wind = starkwind["b) nachts 45 MW"].results.ext_grid_p_mw.sum(axis=1).idxmin()
pr.profil(starkwind, "Abgang 1", wahl=t_wind, titel=f"Starkwind: Abgang 1 am {t_wind:%d.%m. %H:%M}")
pr.abschnitte(starkwind, "Abgang 1", wahl=t_wind)

**Befund Starkwind nachts:**

- **Spannungsprofil Abgang 1** (15.06., 4 Uhr): Trotz Rückspeisung steigt die Spannung an Bus 1 nicht, sie sinkt sogar leicht (Spannungsfall im HS/MS-Trafo 0,6 → 1,0 → 2,0 Prozentpunkte). Im Trafo ist die Reaktanz X viel größer als R: Die Wirkleistung in Gegenrichtung (R · P < 0) hebt kaum an, aber der Blindleistungsbedarf des Trafos wächst mit dem Quadrat des Stroms und senkt die Spannung (X · Q). Die MS-Kabel bleiben unverändert (1,3 Prozentpunkte).

- **a) 30 MW nachts:** Rückspeisung ins 110-kV-Netz bis 18,8 MW, in 59 statt 33 Stunden. Trafo 0-1 ist mit **94 %** höher ausgelastet als bei Höchstlast im Winter. Die MS-Spannung steigt nicht über 1,025 pu, weil der Windpark an der Sammelschiene von Trafo 0-1 einspeist und seine Leistung direkt ins 110-kV-Netz abfließt.
- **b) 45 MW nachts:** Trafo 0-1 wird mit bis zu **155 %** überlastet (42 Stunden), Rückspeisung bis 33,7 MW. Die Spannungen bleiben trotzdem im Band. Engpass ist also die thermische Grenze des Trafos (25 MVA), nicht die Spannung.
- **Prüfungsantwort:** Bei Starkwind und Schwachlast begrenzt der 110/20-kV-Trafo die Einspeisung. Ein größerer Windpark bräuchte einen eigenen Netzanschluss bzw. Umspanner oder Einspeisemanagement (Abregelung bei Überlastung, Redispatch).

## 7 Sturmabschaltung des Windparks

**Frage:** Was passiert, wenn der Windpark bei Volllast wegen Sturm abschaltet (Abschaltgeschwindigkeit überschritten)?

**Annahme:** Am Mittwoch der Januarwoche speist der Windpark von 0 bis 14 Uhr mit 30 MW ein und fällt ab 14 Uhr bis Mitternacht auf 0 MW (zwei einmalige Änderungen).

In [ ]:
tag = pa.typische_wochen().loc["Januar", "Mi"]
sturm = {"Referenz": REF["JANUARY"],
         "Sturm: 30 MW, ab 14 Uhr 0 MW": pr.szenario("sturm", b, monate=["JANUARY"], aenderungen=[
             pa.einmal(WIND, wert_mw=30, datum=tag, von=0, bis=14, name="Volllast"),
             pa.einmal(WIND, wert_mw=0, datum=tag, von=14, bis=24, name="Sturmabschaltung")])}
r = sturm["Sturm: 30 MW, ab 14 Uhr 0 MW"].results
h = pd.date_range(f"{tag} 11:00", periods=6, freq="h")
display(pd.DataFrame({"Wind [MW]": r.sgen_p_mw.loc[h, "Windpark"], "Austausch 110 kV [MW]": r.ext_grid_p_mw.sum(axis=1).loc[h],
                      "Trafo 0-1 [MW] (p_hv)": r.trafo_p_mw.loc[h, "Trafo 0-1"], "Trafo 0-1 [%]": r.trafo_loading.loc[h, "Trafo 0-1"],
                      "U Bus 1 [pu]": r.vm_pu.loc[h, "Bus 1"], "U Bus 11 [pu]": r.vm_pu.loc[h, "Bus 11"]},
                     index=h).set_index(h.strftime("%H:%M")).round(3))
pr.verlauf(sturm, f"Sturmabschaltung am {tag} um 14 Uhr", tage=[tag])

**Spannungsprofil** in Abgang 1 direkt vor und nach der Abschaltung (13 und 14 Uhr, dieselbe Simulation):

In [ ]:
ss = sturm["Sturm: 30 MW, ab 14 Uhr 0 MW"]
vor_nach = {"13 Uhr (30 MW Wind)": ss, "14 Uhr (Wind aus)": ss}
w = {"13 Uhr (30 MW Wind)": f"{tag} 13:00", "14 Uhr (Wind aus)": f"{tag} 14:00"}
pr.profil(vor_nach, "Abgang 1", wahl=w, titel=f"Sturmabschaltung am {tag}: Abgang 1 vor und nach 14 Uhr")
pr.abschnitte(vor_nach, "Abgang 1", wahl=w)

**Befund Sturmabschaltung:**

- **Spannungsprofil Abgang 1** (13 und 14 Uhr): fast deckungsgleich. Der Fall im HS/MS-Trafo steigt um 0,2 Prozentpunkte, auf den Kabeln ändert sich nichts. Bus 11 bleibt bei 0,967 pu.

- Um 14 Uhr fallen 30 MW weg: Trafo 0-1 springt von **14 MW Rückspeisung auf 15 MW Bezug**, der Austausch mit dem 110-kV-Netz von −2 auf +28 MW. Die Auslastung von Trafo 0-1 bleibt dabei fast gleich (58 → 62 %), weil nur die Richtung wechselt.
- Die Spannungen ändern sich kaum (Bus 1: 1,010 → 1,008 pu, Bus 11: 0,968 → 0,967 pu). Der Zustand danach ist der Normalbetrieb ohne Wind.
- **Prüfungsantwort:** Für das Verteilnetz ist die Sturmabschaltung unkritisch. Das Problem liegt im überlagerten Netz: 30 MW müssen schlagartig anders bereitgestellt werden (Regelleistung). Bei vielen Windparks gleichzeitig ist das ein Thema der Systemstabilität (Frequenz).

## 8 Sonniger Sonntag mit niedriger Last

**Frage:** Was passiert an einem sonnigen Sonntag, an dem viele Haushalte verreist sind? Und mit deutlich mehr PV-Haushalten (Variante „Zukunft“ aus Teil 4b)?

**Annahme (Juliwoche):** Am Sonntag verbrauchen die Haushalte im Wohngebiet nur 60 % (**a**). In **b** und **c** dasselbe Netz mit 50 % PV-Haushalten (40 % mit Speicher) statt 15 %.

In [ ]:
bz = pr.basis("Zukunft")
sonntag = [pa.einmal(HAUSHALTE, faktor=0.6, monat="JULY", wochentag=6, name="Sonntag 60 % Last")]
so_tag = pa.typische_wochen().loc["Juli", "So"]
sonntag_sims = {"Referenz (heute)": REF["JULY"],
                "a) Sonntag 60 % Last (heute)": pr.szenario("sonntag_heute", b, monate=["JULY"], aenderungen=sonntag),
                "b) Zukunft, normale Last": pr.szenario("zukunft_ref", bz, monate=["JULY"]),
                "c) Zukunft, Sonntag 60 % Last": pr.szenario("sonntag_zukunft", bz, monate=["JULY"], aenderungen=sonntag)}
tab = pr.kennzahlen(sonntag_sims)
tab

In [ ]:
pr.verlauf(sonntag_sims, f"Sonntag {so_tag}: niedrige Last, heute und Zukunft", tage=[so_tag])

**Spannungsprofil im Wohngebiet** zur Stunde mit der größten Spannungsanhebung (Rückspeisung):

In [ ]:
pr.profil(sonntag_sims, "Wohngebiet", wahl="steig", titel="Sonniger Sonntag: Wohngebiet in der Stunde mit der größten Anhebung")
display(pr.abschnitte(sonntag_sims, "Wohngebiet", wahl="steig"))
pr.spannung_zeit_ort(sonntag_sims["b) Zukunft, normale Last"], "Wohngebiet", monat="JULY", vmax=1.2,
                     titel="Zukunft: Spannung im Wohngebiet über die Woche")

**Befund sonniger Sonntag:**

- **Spannungsprofil** (25.07., 13 Uhr): Heute steigt die Spannung auf den NS-Kabeln um 4,9 Prozentpunkte, in der Variante Zukunft um **19,6 Prozentpunkte**: Vom Ortsnetztrafo (Stufe +2, senkt um 3,9 Prozentpunkte) bis R15 klettert sie von unter 1,0 auf 1,17 pu. Das Farbbild zeigt, dass das jeden Mittag passiert und vom Strangende her nach oben wächst. Weniger Last am Sonntag ändert das Profil nicht sichtbar.

- **Heute:** Weniger Last am Sonntag ändert wenig: Rückspeisung bis 375 statt 360 kW, U max 1,049 pu. Der rONT hält die Spannung im Band. Die PV-Leistung ist mittags ohnehin viel größer als der Verbrauch, 40 % weniger Last machen nur einen kleinen Teil davon aus.
- **Zukunft (50 % PV-Haushalte):** Schon an normalen Tagen speist das Wohngebiet bis **1,47 MW** zurück. Ortsnetztrafo R wird mit bis zu **247 %** belastet, Kabel R2-R3 mit 245 %, in 43 Stunden überlastet. Die Spannung an R15 steigt auf **1,17 pu**, in 24 Stunden über 1,10 pu, obwohl der rONT auf der höchsten Stufe steht.
- **Prüfungsantwort:** Der kritische Fall im Wohngebiet ist Sonne bei geringer Last, nicht Höchstlast. Mit wachsendem PV-Anteil reicht der rONT nicht mehr; nötig sind netzdienliche Speicher (laden erst zur Mittagsspitze), Wirkleistungsbegrenzung der Wechselrichter, Q(U)-Regelung oder Netzausbau.

## 9 S1 wird bei viel Wind geschlossen

**Frage:** Was passiert, wenn der Netzbetreiber den Trennschalter S1 schließt, während viel Wind weht?

**Annahme:** Dezemberwoche (windreich), S1 dauerhaft geschlossen, sonst unverändert.

In [ ]:
s1 = {"Referenz (S1 offen)": REF["DECEMBER"],
      "S1 geschlossen": pr.szenario("s1_zu", b, monate=["DECEMBER"], netz=lambda s: s.changeParameter("S1", "closed", True))}
tab = pr.kennzahlen(s1)
display(tab)
r0, r1 = (s.results for s in s1.values())
richtung = {}
for l in ("Line 1-2", "Line 2-3", "Line 3-8", "Line 14-8", "Line 12-13", "Line 13-14"):
    richtung[l] = {"S1 offen: h mit p_from > 0": int((r0.line_p_mw[l] > 0.001).sum()), "S1 offen: h mit p_from < 0": int((r0.line_p_mw[l] < -0.001).sum()),
                   "S1 zu: h mit p_from > 0": int((r1.line_p_mw[l] > 0.001).sum()), "S1 zu: h mit p_from < 0": int((r1.line_p_mw[l] < -0.001).sum()),
                   "max. Auslastung S1 offen [%]": r0.line_loading[l].max(), "max. Auslastung S1 zu [%]": r1.line_loading[l].max()}
print("Flussrichtung (p_from > 0: vom ersten zum zweiten Bus im Namen) und Auslastung:")
pd.DataFrame(richtung).T.round(1)

In [ ]:
pr.verlauf(s1, "S1 geschlossen bei viel Wind (Dezemberwoche)", monat="DECEMBER")

**Spannungsprofil** in beiden Abgängen zur Stunde mit der niedrigsten Spannung an Bus 11:

In [ ]:
pr.profil(s1, "Abgang 1", wahl="fall", titel="S1 offen/zu: Abgang 1")
display(pr.abschnitte(s1, "Abgang 1", wahl="fall"))
t = pr.gemeinsame_stunde(s1, "Bus 11", "fall")
pr.abschnitte(s1, "Abgang 2", wahl=t)

**Befund S1 bei viel Wind:**

- **Spannungsprofil** (19.12., 18 Uhr): Mit S1 halbiert sich fast der Spannungsfall auf den Kabeln von Abgang 1 (5,2 → 3,0 Prozentpunkte, Bus 11 0,952 → 0,977 pu), weil Bus 8 jetzt von zwei Seiten gespeist wird. Abgang 2 übernimmt dafür Last: sein Kabelfall steigt von 0,5 auf 2,1 Prozentpunkte, Bus 14 sinkt von 1,002 auf 0,983 pu. S1 verteilt den Spannungsfall gleichmäßiger auf beide Abgänge.

- **Auslastung:** Line 1-2 und 2-3 steigen von 77 auf **97,5 %**, Line 3-8 von 36 auf 66 %. Der Windstrom fließt jetzt auch durch Abgang 1 und über die Kupplung (Line 14-8) zu Trafo 0-12. Trafo 0-1 wird entlastet (88,5 → 78,5 %), Trafo 0-12 stärker belastet (70 → 76 %).
- **Flussrichtung:** Auf der Kupplung und in Abgang 2 (Line 12-13, 13-14) wechselt die Richtung je nach Wind (Tabelle). Bei offenem S1 fließt dort immer Leistung vom Umspannwerk weg.
- **Spannung:** Abgang 1 wird angehoben (U min 0,976 statt 0,952 pu), weil er von zwei Seiten versorgt wird.
- **Verluste:** + 63 % (16,8 → 27,3 MWh), weil Leistung über den langen Ring fließt.
- **Prüfungsantwort:** S1 verbessert die Spannung, kostet aber Verluste und bringt Line 1-2 an die Grenze. Mit Windpark bleibt S1 im Normalbetrieb offen.

## 10 Ausfall von Line 2-3

**Frage:** Was passiert, wenn Line 2-3 ausfällt?

**Annahme:** Januarwoche, Line 2-3 außer Betrieb. **a)** ohne Wiederversorgung: Bus 3 bis 11 sind getrennt (sofern nicht über S2/S3 versorgt). **b)** mit Wiederversorgung nach dem Plan aus Teil 3b/4a: S1 wird geschlossen.

In [ ]:
def line23(s):
    s.changeParameter("Line 2-3", "in_service", False)

def line23_s1(s):
    line23(s)
    s.changeParameter("S1", "closed", True)

leitung = {"Referenz": REF["JANUARY"],
           "a) Line 2-3 aus": pr.szenario("line23_aus", b, monate=["JANUARY"], netz=line23),
           "b) Line 2-3 aus, S1 zu": pr.szenario("line23_s1", b, monate=["JANUARY"], netz=line23_s1)}
r = leitung["a) Line 2-3 aus"].results
print("ohne Wiederversorgung nicht versorgte Busse (Spannung NaN):",
      [c for c in r.vm_pu.columns if r.vm_pu[c].isna().all() and not c.startswith("LV")])
tab = pr.kennzahlen(leitung)
tab

In [ ]:
pr.verlauf({k: leitung[k] for k in ("Referenz", "b) Line 2-3 aus, S1 zu")}, "Ausfall Line 2-3, Wiederversorgung über S1", monat="JANUARY")

**Spannungsprofil** bis Bus 11: normal über Line 1-2/2-3, nach dem Ausfall über Abgang 2 und die Kupplung (je Variante eigene Zeile, weil der Weg ein anderer ist):

In [ ]:
lv = {k: leitung[k] for k in ("Referenz", "b) Line 2-3 aus, S1 zu")}
pr.profil(lv, "Abgang 1", wahl="min", titel="Ausfall Line 2-3: Weg der Versorgung zu Bus 11")
pr.abschnitte(lv, "Abgang 1", wahl="min")

**Befund Ausfall Line 2-3:**

- **Spannungsprofil** (10.01., 18 Uhr): Bus 11 wird über Bus 12 → 13 → 14 → 8 versorgt (gut 11 km). Der HS/MS-Trafo 0-12 verliert 3,1 statt 2,4 Prozentpunkte, die Kabel 6,5 statt 5,3 Prozentpunkte; Bus 11 liegt bei 0,933 statt 0,953 pu. Der tiefste Punkt ist Bus 6 am Ende der Masche über Bus 3 (0,926 pu).

- **a) ohne Wiederversorgung:** Bus 3 bis 11 sind ohne Versorgung (Spannung NaN). Der Rest des Netzes läuft normal weiter; die Kennzahlen gelten nur für die versorgten Busse.
- **b) Wiederversorgung über S1:** Bus 3 bis 11 werden über Abgang 2 und die Kupplung versorgt. Das funktioniert: keine Überlastung (Trafo 0-12 bis 87 %, Line 12-13 bis 66 %), aber die Spannung fällt am Ende des langen Wegs auf **0,926 pu an Bus 6**, in 70 Stunden unter 0,95 pu. Die Arbeitsgrenze 0,90 pu wird eingehalten. Die rONT schalten häufiger (38 statt 24), weil Bus 12 tiefer liegt.
- **Prüfungsantwort:** Der (n-1)-Fall ist beherrschbar, aber nur im Arbeitsband, nicht im Planungsband. Die Verluste steigen um gut 50 %.

## 11 Ausfall von Trafo 0-1

**Frage:** Was passiert, wenn der 110/20-kV-Trafo von Abgang 1 ausfällt?

**Annahme:** Januarwoche, Trafo 0-1 außer Betrieb, S1 geschlossen (einzige Möglichkeit, Abgang 1 zu versorgen). **a)** Windpark speist weiter ein, **b)** Windpark abgeschaltet.

In [ ]:
def trafo01(s):
    s.changeParameter("Trafo 0-1", "in_service", False)
    s.changeParameter("S1", "closed", True)

def trafo01_ohne_wind(s):
    trafo01(s)
    s.changeParameter("Windpark", "in_service", False)

trafo = {"Referenz": REF["JANUARY"],
         "a) Trafo 0-1 aus, S1 zu": pr.szenario("trafo01_aus", b, monate=["JANUARY"], netz=trafo01),
         "b) zusätzlich Windpark aus": pr.szenario("trafo01_aus_ohne_wind", b, monate=["JANUARY"], netz=trafo01_ohne_wind)}
tab = pr.kennzahlen(trafo)
tab

In [ ]:
pr.verlauf(trafo, "Ausfall Trafo 0-1 (Lücken = Lastfluss nicht konvergiert)", monat="JANUARY")

**Spannungsprofil** bis Bus 11 nach dem Ausfall (Versorgung über Trafo 0-12, Abgang 2 und die Kupplung), in der Stunde mit der niedrigsten und der höchsten Spannung an Bus 11:

In [ ]:
tr = {"Trafo 0-1 aus, Windpark an": trafo["a) Trafo 0-1 aus, S1 zu"], "Trafo 0-1 aus, Windpark aus": trafo["b) zusätzlich Windpark aus"]}
pr.profil(tr, "Abgang 1", wahl="min_je", titel="Ausfall Trafo 0-1: niedrigste Spannung an Bus 11 (je Variante)")
pr.profil({"Trafo 0-1 aus, Windpark an": tr["Trafo 0-1 aus, Windpark an"]}, "Bus 1", wahl="max",
          titel="Ausfall Trafo 0-1, Windpark an: höchste Spannung an Bus 1 (Windpark)")

**Befund Ausfall Trafo 0-1:**

- **Spannungsprofil:** Mit Windpark steigt die Spannung vom Umspannwerk zum Windpark hin um **27 Prozentpunkte** an (Bus 12 0,975 pu → Bus 1 1,245 pu): Die Windleistung muss über rund 18 km Kabel in Gegenrichtung zu Trafo 0-12 fließen. Ohne Windpark fällt die Spannung in der anderen Richtung bis zum Einbruch.

- **a) Windpark speist weiter:** In 45 von 168 Stunden konvergiert der Lastfluss nicht. Bei Wind hebt der Windpark Bus 1 bis **1,25 pu** an, Line 1-2 wird mit bis zu **379 %** belastet, Trafo 0-12 mit 130 %. Bei wenig Wind bricht die Spannung bis 0,52 pu ein.
- **b) Windpark abgeschaltet:** Die Überspannung verschwindet, aber jetzt muss die ganze Last von Abgang 1 über den langen Weg fließen: In **138 Stunden** keine Lösung, sonst Spannung bis 0,59 pu und Line 3-8 mit 259 %.
- **Prüfungsantwort:** Die Kupplung am Ende der Abgänge kann keinen ganzen Abgang übernehmen. Der Ausfall eines 110/20-kV-Trafos ist mit S1 nicht beherrschbar; nötig wäre eine Kupplung im Umspannwerk (beide Trafos auf eine Sammelschiene) oder ein dritter Trafo.

## 12 Der Stufenschalter fällt aus

**Frage:** Was passiert, wenn die rONT nicht mehr regeln (Stufenschalter klemmt in Mittelstellung)?

**Annahme:** Alle drei Ortsnetztrafos bleiben auf Stufe 0, im Winter (Januar, Abendspitze) und im Sommer (Juli, Rückspeisung).

In [ ]:
ront = {"Januar: mit rONT": REF["JANUARY"],
        "Januar: Stufe fest 0": pr.szenario("ohne_ront_jan", b, monate=["JANUARY"], regler=False),
        "Juli: mit rONT": REF["JULY"],
        "Juli: Stufe fest 0": pr.szenario("ohne_ront_jul", b, monate=["JULY"], regler=False)}
tab = pr.kennzahlen(ront)
tab

In [ ]:
pr.verlauf({k: ront[k] for k in list(ront)[:2]}, "Stufenschalter fest – Januar", monat="JANUARY")
pr.verlauf({k: ront[k] for k in list(ront)[2:]}, "Stufenschalter fest – Juli", monat="JULY")

**Spannungsprofil im Wohngebiet** mit und ohne Regelung, im Winter zur niedrigsten und im Sommer zur höchsten Spannung am Strangende:

In [ ]:
pr.profil({k: ront[k] for k in list(ront)[:2]}, "Wohngebiet", wahl="min", titel="Januar: mit rONT / Stufe fest")
display(pr.abschnitte({k: ront[k] for k in list(ront)[:2]}, "Wohngebiet", wahl="min"))
pr.profil({k: ront[k] for k in list(ront)[2:]}, "Wohngebiet", wahl="max", titel="Juli: mit rONT / Stufe fest")
pr.abschnitte({k: ront[k] for k in list(ront)[2:]}, "Wohngebiet", wahl="max")

**Befund Stufenschalter fest:**

- **Spannungsprofil:** Der Spannungsfall auf den NS-Kabeln ist mit und ohne Regelung gleich (Januar 5,8 bis 5,9, Juli −4,9 bis −4,8 Prozentpunkte). Der Unterschied entsteht allein im Ortsnetztrafo: Mit Regelung hebt er im Winter um 0,7 Prozentpunkte an statt 2,0 zu verlieren, im Sommer senkt er um 2,4 Prozentpunkte ab statt nichts zu tun. Der rONT verschiebt das ganze Profil des Strangs nach oben oder unten, seine Form ändert er nicht.

- **Januar:** Ohne Regelung fällt die Spannung an R15 zur Abendspitze auf **0,928 pu** (unter das Planungsband, über der Arbeitsgrenze).
- **Juli:** Mittags steigt sie bei Rückspeisung auf **1,065 pu**.
- Im MS-Netz ändert sich nichts.
- **Prüfungsantwort:** Der rONT regelt jetzt in beide Richtungen. Fällt er aus, liegt das Wohngebiet im Winter zu tief und im Sommer zu hoch, die Arbeitsgrenzen 0,90 … 1,10 pu werden heute aber noch eingehalten.

## 13 Windpark mit Blindleistungsbezug

**Frage:** Was passiert, wenn der Windpark mit cos φ = 0,95 untererregt betrieben wird (Blindleistung aufnimmt)?

**Annahme:** Januarwoche, Windpark cos φ = −0,95 (untererregt), sonst unverändert.

In [ ]:
blind = {"Referenz (cos φ = 1)": REF["JANUARY"],
         "cos φ = 0,95 untererregt": pr.szenario("wind_cosphi", b, monate=["JANUARY"],
             netz=lambda s: s.changeParameter("Windpark", "cos_phi", -0.95))}
tab = pr.kennzahlen(blind)
display(tab)
r0, r1 = (s.results for s in blind.values())
pd.DataFrame({"Q Windpark max [Mvar]": [-r0.sgen_q_mvar["Windpark"].min(), -r1.sgen_q_mvar["Windpark"].min()],
              "Q aus 110 kV max [Mvar]": [r0.ext_grid_q_mvar.sum(axis=1).max(), r1.ext_grid_q_mvar.sum(axis=1).max()]},
             index=list(blind)).round(2)

In [ ]:
pr.verlauf(blind, "Windpark mit cos φ = 0,95 untererregt (Januarwoche)", monat="JANUARY")

**Spannungsprofil** in Abgang 1 zur Stunde mit der niedrigsten Spannung an Bus 11:

In [ ]:
pr.profil(blind, "Abgang 1", wahl="min", titel="Blindleistung: Abgang 1")
pr.abschnitte(blind, "Abgang 1", wahl="min")

**Befund Blindleistungsbezug:**

- **Spannungsprofil** (06.01., 18 Uhr): Der Spannungsfall im HS/MS-Trafo 0-1 steigt von 2,3 auf **7,4 Prozentpunkte**, auf den Kabeln nur von 5,3 auf 5,6. Die Blindleistung wirkt fast ausschließlich über die Reaktanz des Trafos; Bus 11 fällt auf 0,900 pu.

- Der Windpark nimmt bis **9,9 Mvar** auf, der Bezug aus dem 110-kV-Netz steigt von 11,3 auf 21,7 Mvar.
- Die Spannung in Abgang 1 sinkt: Bus 11 auf **0,900 pu** (Arbeitsgrenze), in 70 Stunden unter 0,95 pu. Trafo 0-1 wird überlastet (bis **103 %**), die Verluste steigen um 8 %.
- **Prüfungsantwort:** Untererregter Betrieb senkt die Spannung über X · Q. Das ist nur sinnvoll, wenn die Spannung zu hoch ist. In diesem Netz ist sie es nicht, deshalb schadet es. Besser: cos φ = 1 oder eine Q(U)-Kennlinie, die nur bei hoher Spannung Blindleistung aufnimmt.

## 14 Spannung im 110-kV-Netz ändert sich

**Frage:** Was passiert, wenn das überlagerte 110-kV-Netz nicht mit 1,03 pu, sondern mit 1,00 pu bzw. 1,05 pu betrieben wird?

**Annahme:** Spannung der Netzeinspeisung (`ext_grid`, im Referenznetz 1,03 pu) auf 1,00 pu im Januar (Höchstlast) und auf 1,05 pu im Juli (Rückspeisung).

In [ ]:
EXT = b.net.ext_grid["name"].iloc[0]
hs = {"Januar: 1,03 pu (Referenz)": REF["JANUARY"],
      "Januar: 1,00 pu": pr.szenario("hs_100", b, monate=["JANUARY"], netz=lambda s: s.changeParameter(EXT, "vm_pu", 1.00)),
      "Juli: 1,03 pu (Referenz)": REF["JULY"],
      "Juli: 1,05 pu": pr.szenario("hs_105", b, monate=["JULY"], netz=lambda s: s.changeParameter(EXT, "vm_pu", 1.05))}
tab = pr.kennzahlen(hs)
tab

In [ ]:
pr.verlauf({k: hs[k] for k in list(hs)[:2]}, "110-kV-Spannung 1,00 pu (Januarwoche)", monat="JANUARY")

**Spannungsprofil** bei 1,00 pu im Januar: Abgang 1 und Wohngebiet (der rONT gleicht im Ortsnetz aus):

In [ ]:
hj = {k: hs[k] for k in list(hs)[:2]}
pr.profil(hj, "Abgang 1", wahl="min", titel="110 kV mit 1,00 pu: Abgang 1")
display(pr.abschnitte(hj, "Abgang 1", wahl="min"))
pr.profil(hj, "Wohngebiet", wahl="fall", titel="110 kV mit 1,00 pu: Wohngebiet")
pr.abschnitte(hj, "Wohngebiet", wahl="fall")

**Befund 110-kV-Spannung:**

- **Spannungsprofil:** In Abgang 1 verschiebt sich das ganze Profil um 3 Prozentpunkte nach unten, die Spannungsfälle bleiben fast gleich (Bus 11: 0,952 → 0,918 pu). Im Wohngebiet gleicht der rONT aus: Er hebt um 3,2 statt 0,7 Prozentpunkte an (Stufe −2), R15 bleibt bei 0,950 pu.

- **1,00 pu (Januar):** Das ganze MS-Netz verschiebt sich um etwa 3 Prozentpunkte nach unten: Bus 11 fällt auf **0,918 pu**, in 105 Stunden unter 0,95 pu. Das Ortsnetz bleibt im Band, weil die rONT nachregeln (65 statt 24 Stufenschritte).
- **1,05 pu (Juli):** MS-Netz bis 1,045 pu, das Ortsnetz bleibt durch den rONT bei höchstens 1,049 pu.
- **Prüfungsantwort:** Im MS-Netz gibt es keinen Regler; seine Spannung hängt direkt am überlagerten Netz und an den Trafos 110/20 kV (deren Stufenschalter hier fest sind). Nur das Ortsnetz wird durch die rONT entkoppelt.

## 15 Zusätzliche Einspeisung an Bus 8

**Frage:** Ab welcher zusätzlichen Einspeisung an Bus 8 wird Line 3-8 im Netz am Ende von Teil 4 zum Engpass (vgl. Teil 6, dort im Referenznetz ohne Erzeuger)?

**Annahme:** Juliwoche, zusätzlicher Erzeuger mit konstanter Leistung (Profil FEST) an Bus 8: 2, 4, 6 und 8 MW.

In [ ]:
def bus8(p):
    return lambda s: s.createElement(Generator("Einspeisung Bus 8", bus="Bus 8", profile=Profile.FEST, p_mw=p))

b8 = {"Referenz": REF["JULY"]} | {f"{p} MW an Bus 8": pr.szenario(f"bus8_{p}", b, monate=["JULY"], netz=bus8(p)) for p in (2, 4, 6, 8)}
tab = pr.kennzahlen(b8)
display(tab)
pd.DataFrame({k: {l: s.results.line_loading[l].max() for l in ("Line 3-8", "Line 2-3", "Line 1-2", "Line 8-9")}
              for k, s in b8.items()}).round(1)

**Spannungsprofil** bis Bus 11 (über Bus 8) in der Stunde mit der höchsten Spannung an Bus 8:

In [ ]:
t8 = b8["8 MW an Bus 8"].results.vm_pu["Bus 8"].idxmax()
pr.profil(b8, "Abgang 1", wahl=t8, titel=f"Einspeisung an Bus 8: Abgang 1 am {t8:%d.%m. %H:%M}")
pr.abschnitte(b8, "Abgang 1", wahl=t8)

**Befund Einspeisung an Bus 8:**

- **Spannungsprofil** (24.07., 3 Uhr): Ohne Einspeisung fällt die Spannung auf den Kabeln bis Bus 11 um 1,3 Prozentpunkte. Mit 2 MW an Bus 8 kehrt sich das um, bei 8 MW **steigt** sie vom Umspannwerk bis Bus 8 um 6,2 Prozentpunkte auf 1,084 pu. Bus 8 wird zum Hochpunkt des Abgangs.

- Bei 2 MW sinkt die Auslastung von Line 2-3 und 1-2 zuerst (53 → 18 %), weil die Einspeisung die Last in Abgang 1 deckt. Danach kehrt sich der Fluss um und die Auslastung steigt wieder.
- Line 3-8 erreicht ihre Grenze zwischen 4 und 6 MW (102 % bei 6 MW), bei 8 MW ist sie in allen Stunden überlastet (138 %). Line 2-3 und 1-2 folgen dicht dahinter (bei 6 MW 92 %, bei 8 MW 127 %).
- Die Spannung an Bus 8 steigt auf 1,05 pu (4 MW) bis 1,085 pu (8 MW), also über das Planungsband.
- Im Vergleich zu Teil 6 (6,49 MW bei 60 % Last ohne Erzeuger) liegt die Grenze hier etwas niedriger (knapp 6 MW), weil die Last in Abgang 1 im Juli nachts unter 60 % liegt und weniger von der Einspeisung vor Ort verbraucht wird.
- **Prüfungsantwort:** Eine Einspeisung mitten im Abgang belastet zuerst die Leitung, über die sie abfließt, und hebt dort die Spannung an.

## 16 Heimspeicher

**Frage:** Wie viel tragen die Heimspeicher zur Entlastung des Wohngebiets bei? Was würde ein doppelt so großer Speicher bringen?

**Annahme:** Juliwoche. **a)** alle Heimspeicher außer Betrieb, **b)** Kapazität und Leistung aller Heimspeicher verdoppelt. Die Speicher fahren weiter den Eigenverbrauchsfahrplan (laden, sobald PV-Überschuss da ist).

In [ ]:
def ohne_speicher(s):
    s.net.storage["in_service"] = False

def doppelt(s):
    s.net.storage["max_e_mwh"] *= 2
    s.net.storage["max_p_mw"] *= 2
    s.net.storage["min_p_mw"] *= 2

speicher = {"a) ohne Speicher": pr.szenario("ohne_speicher", b, monate=["JULY"], netz=ohne_speicher),
            "Referenz (314 kWh)": REF["JULY"],
            "b) doppelte Speicher (628 kWh)": pr.szenario("speicher_doppelt", b, monate=["JULY"], netz=doppelt)}
tab = pr.kennzahlen(speicher)
display(tab)
pd.DataFrame({k: {"Rückspeisung Wohngebiet [MWh]": -s.results.trafo_p_mw[pr.ONT["R"]].clip(upper=0).sum(),
                  "Bezug Wohngebiet [MWh]": s.results.trafo_p_mw[pr.ONT["R"]].clip(lower=0).sum(),
                  "Speicher geladen [MWh]": s.results.storage_p_mw.clip(lower=0).sum().sum() if s.results.storage_p_mw.shape[1] else 0.0}
              for k, s in speicher.items()}).round(2)

In [ ]:
pr.verlauf(speicher, "Heimspeicher: ohne, normal, doppelt (Juliwoche)", monat="JULY")

**Spannungsprofil im Wohngebiet** in der Stunde mit der größten Spannungsanhebung:

In [ ]:
pr.profil(speicher, "Wohngebiet", wahl="steig", titel="Heimspeicher: Wohngebiet zur größten Anhebung")
pr.abschnitte(speicher, "Wohngebiet", wahl="steig")

**Befund Heimspeicher:**

- **Spannungsprofil** (25.07., 13 Uhr): in allen drei Fällen identisch. Zur Stunde der größten Spannungsanhebung sind alle Speicher voll und nehmen nichts auf.

- Ohne Speicher speist das Wohngebiet 8,3 MWh zurück, mit Speicher 7,9, mit doppelter Kapazität 7,6 MWh. Die **größte Rückspeisung bleibt in allen Fällen 360 kW**, auch die Spannung (1,049 pu) und die Trafoauslastung ändern sich nicht.
- Grund: Die Speicher laden morgens mit dem ersten Überschuss und sind vor der Mittagsspitze voll. Ein größerer Speicher ist nur etwas später voll.
- **Prüfungsantwort:** Heimspeicher mit Eigenverbrauchsregelung erhöhen den Eigenverbrauch, entlasten das Netz bei der Spitze aber nicht. Netzdienlich wären sie nur, wenn sie erst zur Mittagsspitze laden (mit PV-Prognose) oder eine feste Einspeisegrenze einhalten (Ausblick Teil 9).

## 17 Übersicht

Alle Varianten nebeneinander (jeweils die typische Woche des Szenarios). Grenzwertverletzungen: Spannung außerhalb 0,90 … 1,10 pu oder Auslastung über 100 %.

In [ ]:
alle = {}
for gruppe in (wolken, haushalte, flaute, starkwind, sturm, sonntag_sims, s1, leitung, trafo, ront, blind, hs, b8, speicher):
    for k, s in gruppe.items():
        if not k.lower().startswith(("referenz", "100 %")) and "rONT" not in k and "Referenz" not in k:
            alle[f"{s.name.replace(pr.PREFIX, '')}"] = s
ueb = pr.kennzahlen(alle).T[["davon nicht konvergiert", "U min MS [pu]", "U max MS [pu]", "U min NS [pu]", "U max NS [pu]",
                             "Stunden außerhalb 0,90 … 1,10 (alle)", "Leitung max MS [%]", "Trafo 0-1 max [%]",
                             "Ortsnetztrafo R max [%]", "Stunden mit Überlastung (> 100 %)", "max. Rückspeisung 110 kV [MW]"]]
ueb

**Zusammenfassung für die Prüfung:**

| Szenario | Was wird kritisch? |
|---|---|
| Wolken | kurze Leistungssprünge im Ortsnetz, rONT-Schalthäufigkeit; keine Grenzverletzung |
| mehr Verbrauch | zuerst Ortsnetztrafo und erstes Kabel (über 100 % ab ca. +60 %), dann die Spannung am Strangende |
| Dunkelflaute | nichts im Verteilnetz (Auslegung auf Höchstlast); mehr Bezug aus dem 110-kV-Netz |
| Starkwind nachts | Trafo 0-1 thermisch (94 %, bei 45 MW 155 %) |
| Sturmabschaltung | im Verteilnetz unkritisch, 30 MW Sprung für das überlagerte Netz |
| Sonne + geringe Last | Spannung am Strangende; in der Variante Zukunft Trafo und Kabel weit überlastet |
| S1 zu bei Wind | Line 1-2 an der Grenze, +63 % Verluste |
| Ausfall Line 2-3 | beherrschbar über S1, Spannung bis 0,926 pu |
| Ausfall Trafo 0-1 | nicht beherrschbar (Überspannung bzw. Spannungskollaps) |
| rONT fest | Wohngebiet im Winter zu tief, im Sommer zu hoch |
| cos φ 0,95 untererregt | Abgang 1 bis 0,90 pu, Trafo 0-1 überlastet |
| 110 kV 1,00 pu | MS-Netz bis 0,918 pu; Ortsnetz durch rONT geschützt |
| Einspeisung Bus 8 | Line 3-8 ab gut 4 MW |
| Speicher | kein Einfluss auf die Rückspeisespitze |

Mit dem Werkzeug aus Abschnitt 1 lassen sich weitere Fragen direkt rechnen: neue Variante mit `pr.szenario("name", b, monate=[...], aenderungen=[...], netz=...)`, danach `pr.kennzahlen({...})` und `pr.verlauf({...}, monat=...)`.